# ThuggerDaily and the YSL Trial: Event-Linked Public Narrative Analysis

This notebook generates the quantitative tables and visualizations for a report on whether ThuggerDaily's X/Twitter posts temporally aligned with shifts in public sentiment, discourse volume, engagement, and topic prevalence around key dates in the Young Thug / YSL RICO trial.

**Important framing:** this is an observational public-narrative analysis. The notebook estimates temporal association and ThuggerDaily-aligned influence signals. It does **not** prove that ThuggerDaily caused trial outcomes, legal decisions, guilt, innocence, or public opinion changes.

## Data Inputs

The notebook uses real processed dashboard data generated from the project files under `Data/Cleaned Data/`:

- `data/processed/posts_master.csv`
- `data/processed/thuggerdaily_posts.csv`
- `data/processed/entity_timeseries.csv`
- `data/processed/trial_events.csv`
- `data/processed/topic_assignments.csv`

If processed files need to be rebuilt, run:

```bash
python scripts/build_processed_data.py
```

## Key Trial Dates and Sources

The event list combines the local `Young Thug Time Line.rtf` with web-verified dates. Sources used for key validation include:

- Hip Hop History, *A Timeline of Young Thug and YSL's Atlanta RICO Trial*: https://history.hiphop/young-thug-trial-timeline/
- Pitchfork, lyrics allowed as evidence, November 9, 2023: https://pitchfork.com/news/young-thug-lyrics-can-be-entered-as-evidence-in-ysl-rico-trial-judge-rules/
- The FADER, opening statements, November 27, 2023: https://www.thefader.com/2023/11/27/prosecutor-quotes-the-jungle-book-reads-young-thug-lyrics-in-ysl-trial-opening-statement
- Los Angeles Times, Judge Ural Glanville recused, July 15, 2024: https://www.latimes.com/entertainment-arts/music/story/2024-07-15/ysl-trial-young-thug-judge-recused-ural-glanville-rico
- NPR, Young Thug plea, October 31, 2024: https://www.npr.org/2024/10/31/nx-s1-5174207/young-thug-guilty-plea-ysl-trial
- Los Angeles Times, YSL trial ends, December 3, 2024: https://www.latimes.com/entertainment-arts/music/story/2024-12-03/ysl-rico-trial-ends-defendants-not-guilty-young-thug

Use these sources for report footnotes/citations. The local RTF timeline is used as the project-specific event inventory.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data" / "processed"
FIG = ROOT / "reports" / "figures"
TABLES = ROOT / "reports" / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

posts = pd.read_csv(DATA / "posts_master.csv", parse_dates=["date"])
td = pd.read_csv(DATA / "thuggerdaily_posts.csv", parse_dates=["date"])
entity_ts = pd.read_csv(DATA / "entity_timeseries.csv", parse_dates=["date"])
topics = pd.read_csv(DATA / "topic_assignments.csv", parse_dates=["date"])
trial_events_file = pd.read_csv(DATA / "trial_events.csv", parse_dates=["date"])

for df in [posts, td, entity_ts, topics]:
    if "date" in df:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

print(f"Posts: {len(posts):,}")
print(f"ThuggerDaily posts: {len(td):,}")
print(f"Date range: {posts['date'].min().date()} to {posts['date'].max().date()}")
print(f"Platforms: {posts['platform'].nunique()}")
print(f"Entities: {', '.join(sorted(posts['entity'].dropna().unique()))}")

In [ ]:
# Web-verified and local-RTF-derived event list for the report.
key_events = pd.DataFrame([
    {"event_id": "indictment_arrests", "date": "2022-05-09", "event_name": "YSL indictment and Young Thug arrest", "entity": "Young Thug", "event_type": "legal", "source": "Local RTF; Hip Hop History timeline"},
    {"event_id": "gunna_surrenders", "date": "2022-05-11", "event_name": "Gunna turns himself in", "entity": "Gunna", "event_type": "legal", "source": "Local RTF"},
    {"event_id": "bond_denied", "date": "2022-06-02", "event_name": "Young Thug denied bond", "entity": "Young Thug", "event_type": "legal", "source": "Local RTF; Hip Hop History timeline"},
    {"event_id": "gunna_release", "date": "2022-12-14", "event_name": "Gunna released after Alford plea", "entity": "Gunna", "event_type": "legal", "source": "Local RTF; Hip Hop History timeline"},
    {"event_id": "jury_selection", "date": "2023-01-09", "event_name": "Trial date / jury selection period begins", "entity": "Young Thug", "event_type": "legal", "source": "Local RTF; public trial timelines"},
    {"event_id": "gunna_album", "date": "2023-06-16", "event_name": "Gunna releases A Gift & a Curse", "entity": "Gunna", "event_type": "music", "source": "Local RTF"},
    {"event_id": "thug_album", "date": "2023-06-23", "event_name": "Young Thug releases Business Is Business", "entity": "Young Thug", "event_type": "music", "source": "Local RTF"},
    {"event_id": "lyrics_evidence", "date": "2023-11-09", "event_name": "Lyrics allowed as evidence", "entity": "Young Thug", "event_type": "legal", "source": "Local RTF; Pitchfork"},
    {"event_id": "opening_statements", "date": "2023-11-27", "event_name": "Opening statements begin", "entity": "Young Thug", "event_type": "legal", "source": "Local RTF; The FADER"},
    {"event_id": "lifestyle_played", "date": "2024-01-11", "event_name": "Lifestyle played in court", "entity": "Young Thug", "event_type": "legal/media", "source": "Local RTF"},
    {"event_id": "secret_meeting_contempt", "date": "2024-06-10", "event_name": "Secret meeting dispute and Brian Steel contempt episode", "entity": "Young Thug", "event_type": "legal", "source": "Local RTF; recusal coverage"},
    {"event_id": "trial_paused", "date": "2024-07-01", "event_name": "Trial paused amid recusal dispute", "entity": "Young Thug", "event_type": "legal", "source": "Local RTF"},
    {"event_id": "glanville_recused", "date": "2024-07-15", "event_name": "Judge Ural Glanville recused", "entity": "Young Thug", "event_type": "legal", "source": "Local RTF; Los Angeles Times"},
    {"event_id": "ingram_recused", "date": "2024-07-17", "event_name": "Judge Shukura Ingram recused", "entity": "Young Thug", "event_type": "legal", "source": "Local RTF; Pitchfork case timeline"},
    {"event_id": "bail_denied_again", "date": "2024-07-30", "event_name": "Young Thug bail denied again", "entity": "Young Thug", "event_type": "legal", "source": "Local RTF"},
    {"event_id": "plea_nichols", "date": "2024-10-29", "event_name": "Quamarvious Nichols pleads guilty", "entity": "Young Thug", "event_type": "legal", "source": "Local RTF"},
    {"event_id": "pleas_huey_ryan", "date": "2024-10-30", "event_name": "Marquavius Huey and Rodalius Ryan plead guilty", "entity": "Young Thug", "event_type": "legal", "source": "Local RTF"},
    {"event_id": "young_thug_plea", "date": "2024-10-31", "event_name": "Young Thug pleads guilty and is released on probation terms", "entity": "Young Thug", "event_type": "legal", "source": "Local RTF; NPR"},
    {"event_id": "trial_end", "date": "2024-12-03", "event_name": "YSL trial ends for remaining defendants", "entity": "Young Thug", "event_type": "legal", "source": "Local RTF; Los Angeles Times"},
])
key_events["date"] = pd.to_datetime(key_events["date"])
key_events.to_csv(TABLES / "key_trial_events.csv", index=False)
key_events

In [ ]:
# Data coverage table for the report.
coverage = posts.groupby(["platform", "entity"]).agg(
    n_records=("post_id", "count"),
    mean_sentiment=("sentiment_score", "mean"),
    total_engagement=("total_engagement", "sum"),
    min_date=("date", "min"),
    max_date=("date", "max"),
).reset_index().sort_values("n_records", ascending=False)
coverage.to_csv(TABLES / "coverage_by_platform_entity.csv", index=False)
coverage.head(20)

In [ ]:
# Report figure 1: records by platform and entity.
platform_counts = posts.groupby(["platform", "entity"]).size().reset_index(name="records")
fig = px.bar(
    platform_counts,
    x="platform",
    y="records",
    color="entity",
    template="plotly_white",
    title="Cross-Platform Records by Entity",
)
fig.update_layout(xaxis_title="Platform", yaxis_title="Records", legend_title="Entity")
fig.write_html(FIG / "01_records_by_platform_entity.html")
fig

In [ ]:
# Daily public discourse and ThuggerDaily activity.
daily_public = posts.groupby(["date", "entity"]).agg(
    public_volume=("post_id", "count"),
    mean_sentiment=("sentiment_score", "mean"),
    total_engagement=("total_engagement", "sum"),
).reset_index()

daily_td = td.groupby(["date", "entity"]).agg(
    td_posts=("post_id", "count"),
    td_engagement=("total_engagement", "sum"),
    td_mean_sentiment=("sentiment_score", "mean"),
).reset_index()

fig = px.line(
    daily_public.groupby("date", as_index=False)["public_volume"].sum(),
    x="date",
    y="public_volume",
    template="plotly_white",
    title="Daily Public Discourse Volume Across Platforms",
)
for _, row in key_events.iterrows():
    if row["event_id"] in ["indictment_arrests", "gunna_release", "lyrics_evidence", "opening_statements", "glanville_recused", "young_thug_plea", "trial_end"]:
        fig.add_vline(x=row["date"], line_width=1, line_dash="dash", line_color="#64748b")
fig.update_layout(xaxis_title="Date", yaxis_title="Records per day")
fig.write_html(FIG / "02_public_volume_with_trial_events.html")
fig

In [ ]:
# Report figure 3: sentiment trends by entity with event overlays.
sent_daily = daily_public.copy()
sent_daily["sentiment_14d"] = sent_daily.groupby("entity")["mean_sentiment"].transform(lambda s: s.rolling(14, min_periods=3).mean())
fig = px.line(
    sent_daily,
    x="date",
    y="sentiment_14d",
    color="entity",
    template="plotly_white",
    title="14-Day Smoothed Public Sentiment by Entity",
)
for _, row in key_events.iterrows():
    if row["event_id"] in ["lyrics_evidence", "opening_statements", "glanville_recused", "young_thug_plea", "trial_end"]:
        fig.add_vline(x=row["date"], line_width=1, line_dash="dash", line_color="#64748b")
fig.update_layout(xaxis_title="Date", yaxis_title="Mean sentiment")
fig.write_html(FIG / "03_sentiment_trends_by_entity.html")
fig

In [ ]:
def safe_pct(numer, denom):
    if pd.isna(numer) or pd.isna(denom) or denom == 0:
        return np.nan
    return numer / denom * 100


def window_event_metrics(event_row, entity="Young Thug", window=7):
    event_date = event_row["date"]
    entity_posts = posts[posts["entity"].eq(entity)].copy()
    entity_td = td[td["entity"].eq(entity)].copy()

    pre_mask = entity_posts["date"].between(event_date - pd.Timedelta(days=window), event_date - pd.Timedelta(days=1))
    post_mask = entity_posts["date"].between(event_date, event_date + pd.Timedelta(days=window))
    pre = entity_posts[pre_mask]
    post = entity_posts[post_mask]

    td_post = entity_td[entity_td["date"].between(event_date, event_date + pd.Timedelta(days=window))]
    td_pre = entity_td[entity_td["date"].between(event_date - pd.Timedelta(days=window), event_date - pd.Timedelta(days=1))]

    sentiment_diff = post["sentiment_score"].mean() - pre["sentiment_score"].mean() if len(pre) and len(post) else np.nan
    volume_diff = len(post) - len(pre)
    engagement_diff = post["total_engagement"].sum() - pre["total_engagement"].sum()

    # Attribution-style signal: compare post-window public outcome on days with TD exposure vs post-window days without TD exposure.
    post_daily = post.groupby("date").agg(
        public_volume=("post_id", "count"),
        mean_sentiment=("sentiment_score", "mean"),
        total_engagement=("total_engagement", "sum"),
    ).reset_index()
    td_daily = td_post.groupby("date").agg(td_posts=("post_id", "count"), td_engagement=("total_engagement", "sum")).reset_index()
    merged = post_daily.merge(td_daily, on="date", how="left").fillna({"td_posts": 0, "td_engagement": 0})
    merged["td_exposed"] = merged["td_posts"] > 0
    exposed = merged[merged["td_exposed"]]
    unexposed = merged[~merged["td_exposed"]]

    exposure_volume_lift = exposed["public_volume"].mean() - unexposed["public_volume"].mean() if len(exposed) and len(unexposed) else np.nan
    exposure_sentiment_lift = exposed["mean_sentiment"].mean() - unexposed["mean_sentiment"].mean() if len(exposed) and len(unexposed) else np.nan
    exposure_engagement_lift = exposed["total_engagement"].mean() - unexposed["total_engagement"].mean() if len(exposed) and len(unexposed) else np.nan

    return {
        "event_id": event_row["event_id"],
        "event_name": event_row["event_name"],
        "event_date": event_date,
        "entity": entity,
        "window_days": window,
        "pre_records": len(pre),
        "post_records": len(post),
        "pre_sentiment": pre["sentiment_score"].mean(),
        "post_sentiment": post["sentiment_score"].mean(),
        "sentiment_change": sentiment_diff,
        "volume_change": volume_diff,
        "volume_percent_change": safe_pct(volume_diff, len(pre)),
        "engagement_change": engagement_diff,
        "engagement_percent_change": safe_pct(engagement_diff, pre["total_engagement"].sum()),
        "td_pre_posts": len(td_pre),
        "td_post_posts": len(td_post),
        "td_post_engagement": td_post["total_engagement"].sum(),
        "td_exposed_post_days": int(merged["td_exposed"].sum()) if len(merged) else 0,
        "post_window_days_with_public_data": len(merged),
        "td_exposure_volume_lift": exposure_volume_lift,
        "td_exposure_sentiment_lift": exposure_sentiment_lift,
        "td_exposure_engagement_lift": exposure_engagement_lift,
        "volume_shift_aligned_share": safe_pct(exposure_volume_lift, volume_diff),
        "sentiment_shift_aligned_share": safe_pct(exposure_sentiment_lift, sentiment_diff),
        "engagement_shift_aligned_share": safe_pct(exposure_engagement_lift, engagement_diff),
    }

report_events = key_events[key_events["event_id"].isin([
    "indictment_arrests", "gunna_release", "lyrics_evidence", "opening_statements", "secret_meeting_contempt",
    "glanville_recused", "young_thug_plea", "trial_end"
])]

rows = []
for _, event in report_events.iterrows():
    for entity in ["Young Thug", "Gunna", "YFN Lucci"]:
        rows.append(window_event_metrics(event, entity=entity, window=7))

event_results = pd.DataFrame(rows)
event_results.to_csv(TABLES / "thuggerdaily_event_window_results_7d.csv", index=False)
event_results.sort_values(["event_date", "entity"])

In [ ]:
# Report table: key ThuggerDaily-aligned changes for Young Thug.
young_thug_results = event_results[event_results["entity"].eq("Young Thug")].copy()
summary_cols = [
    "event_date", "event_name", "pre_records", "post_records", "volume_change", "sentiment_change",
    "td_post_posts", "td_post_engagement", "td_exposure_volume_lift", "td_exposure_sentiment_lift",
    "volume_shift_aligned_share", "sentiment_shift_aligned_share"
]
young_thug_results[summary_cols].round(3).to_csv(TABLES / "young_thug_key_event_summary.csv", index=False)
young_thug_results[summary_cols].round(3)

In [ ]:
# Report figure 4: event-window sentiment change by entity.
fig = px.bar(
    event_results,
    x="event_name",
    y="sentiment_change",
    color="entity",
    barmode="group",
    template="plotly_white",
    title="7-Day Pre/Post Sentiment Change Around Key Trial Dates",
)
fig.update_layout(xaxis_title="Event", yaxis_title="Post minus pre mean sentiment", xaxis_tickangle=-35)
fig.write_html(FIG / "04_event_sentiment_change.html")
fig

In [ ]:
# Report figure 5: ThuggerDaily post intensity around key dates.
td_daily_all = td.groupby("date").agg(td_posts=("post_id", "count"), td_engagement=("total_engagement", "sum")).reset_index()
fig = px.bar(
    td_daily_all,
    x="date",
    y="td_posts",
    template="plotly_white",
    title="ThuggerDaily Posting Volume Over Trial Timeline",
)
for _, row in report_events.iterrows():
    fig.add_vline(x=row["date"], line_width=1, line_dash="dash", line_color="#64748b")
fig.update_layout(xaxis_title="Date", yaxis_title="ThuggerDaily posts per day")
fig.write_html(FIG / "05_thuggerdaily_posting_volume.html")
fig

In [ ]:
# Lag correlation: ThuggerDaily posting volume vs downstream public volume/sentiment.
all_dates = pd.date_range(posts["date"].min().date(), posts["date"].max().date(), freq="D")
public_daily_total = posts.groupby("date").agg(public_volume=("post_id", "count"), mean_sentiment=("sentiment_score", "mean"), total_engagement=("total_engagement", "sum")).reindex(all_dates).fillna({"public_volume":0, "total_engagement":0})
public_daily_total.index.name = "date"
td_daily_total = td.groupby("date").agg(td_posts=("post_id", "count"), td_engagement=("total_engagement", "sum")).reindex(all_dates).fillna(0)
td_daily_total.index.name = "date"
lag_rows = []
for lag in [0, 1, 3, 7, 14, 30]:
    lag_rows.append({
        "lag_days": lag,
        "corr_td_posts_public_volume": td_daily_total["td_posts"].corr(public_daily_total["public_volume"].shift(-lag)),
        "corr_td_engagement_public_volume": td_daily_total["td_engagement"].corr(public_daily_total["public_volume"].shift(-lag)),
        "corr_td_posts_sentiment": td_daily_total["td_posts"].corr(public_daily_total["mean_sentiment"].shift(-lag)),
        "corr_td_posts_public_engagement": td_daily_total["td_posts"].corr(public_daily_total["total_engagement"].shift(-lag)),
    })
lag_results = pd.DataFrame(lag_rows)
lag_results.to_csv(TABLES / "lag_correlation_results.csv", index=False)
lag_results.round(3)

In [ ]:
fig = px.line(
    lag_results.melt(id_vars="lag_days", var_name="metric", value_name="correlation"),
    x="lag_days",
    y="correlation",
    color="metric",
    markers=True,
    template="plotly_white",
    title="Exploratory Lag Correlations: ThuggerDaily Activity and Downstream Public Metrics",
)
fig.update_layout(xaxis_title="Lag in days", yaxis_title="Correlation")
fig.write_html(FIG / "06_lag_correlations.html")
fig

In [ ]:
# Topic shifts around key trial dates.
def topic_shift(event_row, entity="Young Thug", window=7):
    event_date = event_row["date"]
    df = posts[posts["entity"].eq(entity)].copy()
    df = df[df["date"].between(event_date - pd.Timedelta(days=window), event_date + pd.Timedelta(days=window))]
    df["period"] = np.where(df["date"] < event_date, "pre", "post")
    counts = df.groupby(["period", "topic_label"]).size().reset_index(name="records")
    counts["share"] = counts["records"] / counts.groupby("period")["records"].transform("sum")
    wide = counts.pivot(index="topic_label", columns="period", values="share").fillna(0)
    wide["share_change"] = wide.get("post", 0) - wide.get("pre", 0)
    wide = wide.reset_index().sort_values("share_change", ascending=False)
    wide["event_id"] = event_row["event_id"]
    wide["event_name"] = event_row["event_name"]
    wide["entity"] = entity
    return wide

topic_rows = []
for _, event in report_events.iterrows():
    topic_rows.append(topic_shift(event, entity="Young Thug", window=7))
topic_results = pd.concat(topic_rows, ignore_index=True)
topic_results.to_csv(TABLES / "young_thug_topic_shift_results.csv", index=False)
topic_results.head(20).round(3)

In [ ]:
# Report figure 7: topic shifts around Young Thug plea.
plea_topics = topic_results[topic_results["event_id"].eq("young_thug_plea")].sort_values("share_change")
fig = px.bar(
    plea_topics,
    x="share_change",
    y="topic_label",
    orientation="h",
    template="plotly_white",
    title="Topic Share Shift Around Young Thug Plea / Release (7-Day Window)",
)
fig.update_layout(xaxis_title="Post minus pre topic share", yaxis_title="Topic")
fig.write_html(FIG / "07_topic_shift_young_thug_plea.html")
fig

## Interpretation Guidance for the Written Report

Use cautious language:

- **Good:** "ThuggerDaily activity was temporally aligned with a downstream increase in public discourse volume."
- **Good:** "The event-window estimate suggests an influence signal, conditional on the observed data and timing."
- **Avoid:** "ThuggerDaily caused the trial outcome."
- **Avoid:** "ThuggerDaily was responsible for public opinion changing" unless you explicitly state the observational assumptions.

Recommended report sections:

1. Data coverage and platform composition.
2. Key YSL trial timeline dates.
3. Public volume and sentiment trends over time.
4. ThuggerDaily posting intensity around key dates.
5. Event-window pre/post results.
6. ThuggerDaily-aligned attribution signals.
7. Topic shifts around major trial events.
8. Limitations: observational design, platform bias, missing denominators, sentiment limitations, omitted confounders.

In [ ]:
# Export a compact report pack as CSVs for LaTeX/report drafting.
report_pack = {
    "records_total": len(posts),
    "thuggerdaily_posts": len(td),
    "date_min": str(posts["date"].min().date()),
    "date_max": str(posts["date"].max().date()),
    "platforms": posts["platform"].nunique(),
    "entities": posts["entity"].nunique(),
    "topics": posts["topic_label"].nunique(),
}
pd.Series(report_pack).to_csv(TABLES / "report_pack_summary.csv", header=["value"])
report_pack